In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# SAM 3 Agent

This notebook shows an example of how an MLLM can use SAM 3 as a tool, i.e., "SAM 3 Agent", to segment more complex text queries such as "the leftmost child wearing blue vest".

## Env Setup

First install `sam3` in your environment using the [installation instructions](https://github.com/facebookresearch/sam3?tab=readme-ov-file#installation) in the repository.

In [2]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()

In [3]:
import os

# Derive the repo root from the installed `sam3` package instead of walking up
# from os.getcwd(). This is idempotent: re-running this cell won't keep moving the
# working directory up a level (the previous `os.path.dirname(os.getcwd())` broke
# all relative paths, e.g. assets/images/..., if the cell ran more than once).
import sam3

SAM3_ROOT = os.path.abspath(os.path.join(os.path.dirname(sam3.__file__), ".."))
os.chdir(SAM3_ROOT)
print("working dir:", os.getcwd())

# setup GPU to use -  A single GPU is good with the purpose of this demo
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
_ = os.system("nvidia-smi")


/home/albert/Desktop/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


working dir: /home/albert/Desktop/sam3
Fri Jul 10 10:37:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:09:00.0  On |                  N/A |
| 37%   36C    P5             46W /  420W |   16199MiB /  24576MiB |     26%      Default |
|                                         |                        |                  N/A |
+--------

/home/albert/.local/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Build SAM3 Model

In [4]:
import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam3_root = os.path.dirname(sam3.__file__)
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)
processor = Sam3Processor(model, confidence_threshold=0.5)

## LLM Setup

Config which MLLM to use, it can either be a model served by vLLM that you launch from your own machine or a model is served via external API. If you want to using a vLLM model, we also provided insturctions below.

In [5]:
# ---------------------------------------------------------------------------
# CHOOSING THE AGENT LLM
#
# This agent drives a LONG (~66 KB) reasoning/tool protocol. It was built for a
# frontier VLM (the code default is Llama-4-Maverick-17B-128E). Small models that
# fit on a single 24 GB GPU (e.g. Qwen3-VL-8B) DO run, and memory is now bounded,
# but they are NOT reliably strong enough to emit the required <tool>/<verdict>
# JSON — expect malformed tool calls. For real agent behavior, point at a hosted
# frontier VLM via an OpenAI-compatible API (the "external" option below).
# ---------------------------------------------------------------------------
LLM_CONFIGS = {
    # --- Local vLLM (fits on one 24 GB GPU; good for smoke-testing the plumbing) ---
    # Serve with:
    #   vllm serve Qwen/Qwen3-VL-8B-Instruct --tensor-parallel-size 1 \
    #     --quantization fp8 --max-model-len 24576 --gpu-memory-utilization 0.70 \
    #     --allowed-local-media-path / --enforce-eager --port 8002
    "qwen3_vl_8b_instruct": {
        "provider": "vllm",
        "model": "Qwen/Qwen3-VL-8B-Instruct",
    },
    # --- External API (RECOMMENDED for real agent runs) ---
    # Any OpenAI-compatible endpoint. Fill in base_url + a strong VLM + your key.
    "external_frontier_vlm": {
        "provider": "external",
        "model": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",  # or gpt-4o, etc.
        "base_url": "https://YOUR_OPENAI_COMPATIBLE_ENDPOINT/v1",
    },
}

# Pick one: "qwen3_vl_8b_instruct" (local) or "external_frontier_vlm" (recommended)
model = "qwen3_vl_8b_instruct"
LLM_API_KEY = "DUMMY_API_KEY"  # set your real key when using the external provider

llm_config = LLM_CONFIGS[model]
llm_config["api_key"] = LLM_API_KEY
llm_config["name"] = model

# setup API endpoint
if llm_config["provider"] == "vllm":
    # Port 8002 (8001 was occupied on this box). Match your `vllm serve --port`.
    LLM_SERVER_URL = "http://0.0.0.0:8002/v1"
else:
    LLM_SERVER_URL = llm_config["base_url"]


### Setup vLLM server
This step is only required if you are using a model served by vLLM. Skip it if you
call an LLM via an external API (Gemini, GPT, a hosted frontier VLM, etc.).

* Install vLLM (in a separate conda env from SAM 3 to avoid dependency conflicts).
  ```bash
  conda create -n vllm python=3.12
  pip install vllm --extra-index-url https://download.pytorch.org/whl/cu128
  ```

* Start the vLLM server on the **same single GPU** as this notebook.
  **This box has ONE GPU (24 GB), so `--tensor-parallel-size` must be `1`** (the old
  `--tensor-parallel-size 4` fails with *"World size (4) > available GPUs (1)"*).
  We use fp8 quantization + a capped context so the LLM fits alongside SAM 3, and
  port **8002** (matches `LLM_SERVER_URL` in the config cell above):
  ```bash
  vllm serve Qwen/Qwen3-VL-8B-Instruct \
    --tensor-parallel-size 1 \
    --quantization fp8 \
    --max-model-len 24576 \
    --gpu-memory-utilization 0.70 \
    --allowed-local-media-path / \
    --enforce-eager \
    --port 8002
  ```
  Wait for `Application startup complete` before running the cells below.

> **Heads up on model quality:** the local 8B model runs and no longer blows up
> memory, but it is **not strong enough to reliably drive this agent's protocol**
> (expect malformed tool calls). For correct end-to-end agent behavior, use the
> `external_frontier_vlm` option in the config cell and point it at a hosted
> frontier VLM. See `study/AGENT_NOTES.md` for the full explanation.


## Run SAM3 Agent Inference

In [6]:
from functools import partial
from IPython.display import display, Image
from sam3.agent.client_llm import send_generate_request as send_generate_request_orig
from sam3.agent.client_sam3 import call_sam_service as call_sam_service_orig
from sam3.agent.inference import run_single_image_inference

In [ ]:
# prepare input args and run single image inference
image = "assets/images/test_image.jpg"
prompt = "the leftmost child wearing blue vest"
image = os.path.abspath(image)
send_generate_request = partial(send_generate_request_orig, server_url=LLM_SERVER_URL, model=llm_config["model"], api_key=llm_config["api_key"])
call_sam_service = partial(call_sam_service_orig, sam3_processor=processor)
output_image_path = run_single_image_inference(
    image, prompt, llm_config, send_generate_request, call_sam_service,
    debug=True, output_dir="agent_output"
)

# display output
if output_image_path is not None:
    display(Image(filename=output_image_path))

------------------------------ Starting SAM 3 Agent Session... ------------------------------ 
> Text prompt: the leftmost child wearing blue vest
> Image path: /home/albert/Desktop/sam3/assets/images/test_image.jpg



------------------------------ Round 1------------------------------



image_path /home/albert/Desktop/sam3/assets/images/test_image.jpg
🔍 Calling model Qwen/Qwen3-VL-8B-Instruct...


KeyboardInterrupt: 

: 